## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-13-00-52-19 +0000,startribune,Live: Wild take early lead over Maple Leafs as...,https://www.startribune.com/minnesota-wild-tor...
1,2026-11-13-00-51-05 +0000,nypost,LA Mayor Karen Bass offers weak eight-word exc...,https://nypost.com/2026/11/12/us-news/karen-ba...
2,2026-11-13-00-46-55 +0000,nyt,Lutnick Took a Softer Line on Nvidia as Calls ...,https://www.nytimes.com/2026/11/12/us/politics...
3,2026-11-13-00-44-00 +0000,wsj,Some of President Trump’s top advisers have st...,https://www.wsj.com/politics/policy/lutnick-fi...
4,2026-11-13-00-42-59 +0000,nypost,Gavin Newsom reveals his timeline for deciding...,https://nypost.com/2026/11/12/us-news/gavin-ne...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2694/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
40,huang,46
27,nvidia,45
39,jensen,42
178,new,29
33,trump,28


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
266,2026-11-12-14-37-00 +0000,wsj,"From the outset, some Trump administration off...",https://www.wsj.com/politics/national-security...,218
250,2026-11-12-15-50-09 +0000,nypost,Trump floats ex-Apple CEO Tim Cook to succeed ...,https://nypost.com/2026/11/12/us-news/trump-fl...,200
35,2026-11-12-23-34-05 +0000,bbc,"Scandal-hit Nvidia CEO Jensen Huang quits, plu...",https://www.bbc.co.uk/news/articles/cmq5vd1wdg...,188
283,2026-11-12-13-59-07 +0000,chitri,Nvidia CEO Jensen Huang resigns after his affa...,https://www.chicagotribune.com/2026/11/12/nvid...,179
177,2026-11-12-19-09-50 +0000,nypost,California Dem’s wife caught in curious Nvidia...,https://nypost.com/2026/11/12/us-news/rep-ro-k...,179


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
266,218,2026-11-12-14-37-00 +0000,wsj,"From the outset, some Trump administration off...",https://www.wsj.com/politics/national-security...
47,113,2026-11-12-23-05-02 +0000,missionlocal,"Richmond Buzz: New brunch, new pizza, and a ne...",https://missionlocal.org/2026/11/richmond-buzz...
256,99,2026-11-12-15-15-00 +0000,wsj,President Trump said the U.S. doesn’t need a d...,https://www.wsj.com/politics/policy/trump-now-...
379,72,2026-11-12-04-02-00 +0000,wsj,Democrat Scott Colom is the state of Mississip...,https://www.wsj.com/politics/elections/scott-c...
198,65,2026-11-12-18-17-00 +0000,wsj,Pentagon chief Pete Hegseth pinned a medal on ...,https://www.wsj.com/politics/hegseth-and-aides...
242,56,2026-11-12-16-29-00 +0000,wsj,"To win control of the House and Senate, Democr...",https://www.wsj.com/politics/elections/the-vot...
17,54,2026-11-13-00-09-42 +0000,nypost,Texas wife Susan Watford made heartbreaking pl...,https://nypost.com/2026/11/12/us-news/texas-wi...
231,51,2026-11-12-17-08-00 +0000,wsj,Home sales declined in October to their lowest...,https://www.wsj.com/economy/housing/home-sales...
190,49,2026-11-12-18-36-15 +0000,bbc,'How does a building just fall?' Neighbours gr...,https://www.bbc.co.uk/news/articles/cm4gn5qddq...
287,47,2026-11-12-13-47-00 +0000,wsj,Spain’s ACS Expects Data-Center Build-Out to B...,https://www.wsj.com/business/spains-acs-expect...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
